In [1]:
import os
import shutil

### Write ```Dockerfile```

In [2]:
%%writefile Dockerfile

FROM python:3.9
    
# update package manager
RUN apt-get update

# update pip
RUN pip install --upgrade pip

# copy requirements
COPY requirements.txt .

# install dependencies
RUN pip install -r requirements.txt

# copy script into container
COPY script.py .

# run script when image is run
CMD ["python3", "script.py"]

Writing Dockerfile


### Write ```requirements.txt``` to local drive

In [3]:
%%writefile requirements.txt

pyarrow==9.0.0
fsspec==2022.10.0
s3fs==2022.10.0

tqdm==4.64.1
numpy==1.23.4
pandas==1.2.4
scikit_learn==0.24.1
boto3==1.24.59

Writing requirements.txt


### Write ```script.py``` to local drive

In [4]:
%%writefile script.py

import os
import pandas as pd
import numpy as np
import boto3
import pickle

# download from s3
def download_from_s3(str_local_path, str_bucket_path, str_project):
    # download file
    boto3.client('s3').download_file(str_project, str_bucket_path, str_local_path)

# constants
str_project = '20231010-gen-xii'
str_datecol = 'applicationdate__app'
str_target = 'target'
str_dirname_output = './output'
str_model = '03_pricing_lgd'

# create output dir
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass

# get module
print('Downloading module for preprocessing...')
str_filename = 'preprocessing.py'
str_local_path = f'./{str_filename}'
str_bucket_path = f'01_ad/02_model/00_preprocessing/01_create_preprocessor/{str_filename}'
download_from_s3(
    str_local_path=str_local_path, 
    str_bucket_path=str_bucket_path, 
    str_project=str_project,
)

# import preprocessing model
print('Importing preprocessing model...')
str_filename = 'cls_model_preprocessing.pkl'
str_local_path = f'{str_dirname_output}/{str_filename}'
str_bucket_path = f'01_ad/02_model/00_preprocessing/01_create_preprocessor/{str_filename}'
download_from_s3(
    str_local_path=str_local_path, 
    str_bucket_path=str_bucket_path, 
    str_project=str_project,
)
cls_model_preprocessing = pickle.load(open(str_local_path, 'rb'))

# preprocess and save data
for str_df in ['train','valid','test']:
    # import data
    print(f'Importing {str_df} data...')
    str_filename = f'df_{str_df}_noleaks.gzip'
    str_uri = f's3://{str_project}/{str_model}/01_data_prep/05_leaky_features/04_write_dfs/{str_filename}'
    df = pd.read_parquet(str_uri)
    list_target = list(df[str_target])
    # drop target so it doesnt get preprocessed
    df.drop(str_target, axis=1, inplace=True)
    print('')

    # preprocess
    print(f'Preprocessing {str_df} data...')
    df = cls_model_preprocessing.transform(df)
    # re-assign target now that the data has been preprocessed
    df[str_target] = list_target
    print('')
    
    # get non-numeric
    list_non_numeric = []
    for col in df.columns:
        if df[col].dtype not in ['float64','int64']:
            list_non_numeric.append(col)
    # rm date col
    list_non_numeric = [col for col in list_non_numeric if col != str_datecol]
    # rm target
    list_non_numeric = [col for col in list_non_numeric if col != str_target]
    
    # non-numeric to string
    print('Converting non-numeric to string...')
    df[list_non_numeric] = df[list_non_numeric].astype(str)
    
    # write full data set to s3
    print(f'Writing {str_df} data to s3...')
    str_filename = f'df_{str_df}_noleaks_pre.gzip'
    str_uri = f's3://{str_project}/{str_model}/02_model/00_preprocessing/02_make_dfs/{str_filename}'
    df.to_parquet(str_uri, compression='gzip')
    
    # logic for writing samples of training data
    if str_df == 'train':
        # write sample to s3
        print('Writing 75% sample to s3...')
        df = df.sample(frac=0.75, random_state=42) # 75% of 100%
        str_filename = f'df_{str_df}_noleaks_pre_75.gzip'
        str_uri = f's3://{str_project}/{str_model}/02_model/00_preprocessing/02_make_dfs/{str_filename}'
        df.to_parquet(str_uri, compression='gzip')
        print('')
        
        # write sample to s3
        print('Writing 50% sample to s3...')
        df = df.sample(frac=0.666, random_state=42) # 66% of 75% is 50% of 100%
        str_filename = f'df_{str_df}_noleaks_pre_50.gzip'
        str_uri = f's3://{str_project}/{str_model}/02_model/00_preprocessing/02_make_dfs/{str_filename}'
        df.to_parquet(str_uri, compression='gzip')
        print('')
        
        # write sample to s3
        print('Writing 25% sample to s3...')
        df = df.sample(frac=0.50, random_state=42) # 50% of 66.6% is 25% of 100%
        str_filename = f'df_{str_df}_noleaks_pre_25.gzip'
        str_uri = f's3://{str_project}/{str_model}/02_model/00_preprocessing/02_make_dfs/{str_filename}'
        df.to_parquet(str_uri, compression='gzip')
        print('')
        
        # write sample to s3
        print('Writing 1% sample to s3...')
        df = df.sample(frac=0.04, random_state=42) # 4% of 25% is 1% of 100%
        str_filename = f'df_{str_df}_noleaks_pre_1.gzip'
        str_uri = f's3://{str_project}/{str_model}/02_model/00_preprocessing/02_make_dfs/{str_filename}'
        df.to_parquet(str_uri, compression='gzip')
    else:
        pass

Writing script.py


### Build and push to ECR

In [5]:
%%sh

# name the image
image=genxii-lgd-make-dfs

# build image
docker build -t ${image} .

# get region
region=$(aws configure get region)
region=${region:-us-west-2}

# get account
account=$(aws sts get-caller-identity --query Account --output text)

# get full name
fullname="${account}.dkr.ecr.${region}.amazonaws.com/${image}:latest"

# get login command and execute it
aws ecr get-login-password --region "${region}" | docker login --username AWS --password-stdin "${account}".dkr.ecr."${region}".amazonaws.com

# create repository in ECR
aws ecr create-repository --repository-name "${image}" --image-scanning-configuration scanOnPush=true --image-tag-mutability MUTABLE

# tag image
docker tag  ${image} ${fullname}

# push image to ECR   
docker push ${fullname}

Sending build context to Docker daemon  23.04kB
Step 1/7 : FROM python:3.9
3.9: Pulling from library/python
609c73876867: Pulling fs layer
7247ea8d81e6: Pulling fs layer
be374d06f382: Pulling fs layer
b4580645a8e5: Pulling fs layer
aa7e0aca67dd: Pulling fs layer
75dfb2afacda: Pulling fs layer
9005ef15c559: Pulling fs layer
03a1dbecb08c: Pulling fs layer
9005ef15c559: Waiting
03a1dbecb08c: Waiting
b4580645a8e5: Waiting
aa7e0aca67dd: Waiting
75dfb2afacda: Waiting
7247ea8d81e6: Verifying Checksum
7247ea8d81e6: Download complete
609c73876867: Verifying Checksum
609c73876867: Download complete
be374d06f382: Verifying Checksum
be374d06f382: Download complete
aa7e0aca67dd: Verifying Checksum
aa7e0aca67dd: Download complete
75dfb2afacda: Verifying Checksum
75dfb2afacda: Download complete
9005ef15c559: Verifying Checksum
9005ef15c559: Download complete
03a1dbecb08c: Verifying Checksum
03a1dbecb08c: Download complete
609c73876867: Pull complete
7247ea8d81e6: Pull complete
b4580645a8e5: Verifying

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.8/138.8 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 126.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.5/132.5 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 85.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 114.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 301.2/301.2 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.9/229.9 kB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 505.

WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store



Login Succeeded



An error occurred (RepositoryAlreadyExistsException) when calling the CreateRepository operation: The repository with name 'genxii-lgd-make-dfs' already exists in the registry with id '836690756591'


The push refers to repository [836690756591.dkr.ecr.us-west-2.amazonaws.com/genxii-lgd-make-dfs]
abe7a43c6225: Preparing
5df1cef74db0: Preparing
7c4f65601b84: Preparing
4db3614013e6: Preparing
f3ccc02fdea4: Preparing
78ecb2a2f011: Preparing
84062ebc4cf5: Preparing
2180aea5f54b: Preparing
86388e04a96b: Preparing
893507f6057f: Preparing
2353f7120e0e: Preparing
51a9318e6edf: Preparing
c5bb35826823: Preparing
2180aea5f54b: Waiting
78ecb2a2f011: Waiting
86388e04a96b: Waiting
893507f6057f: Waiting
2353f7120e0e: Waiting
51a9318e6edf: Waiting
84062ebc4cf5: Waiting
c5bb35826823: Waiting
7c4f65601b84: Pushed
abe7a43c6225: Pushed
84062ebc4cf5: Pushed
78ecb2a2f011: Pushed
4db3614013e6: Pushed
f3ccc02fdea4: Pushed
86388e04a96b: Pushed
2180aea5f54b: Pushed
51a9318e6edf: Pushed
c5bb35826823: Pushed
2353f7120e0e: Pushed
893507f6057f: Pushed
5df1cef74db0: Pushed
latest: digest: sha256:4aa2911931c4d5de42dc3bd21bf88094c87b2c4da2f8f4d1074a7efa3caa11fe size: 3058


### Clean-up

In [6]:
# rm files
for str_file in ['Dockerfile','requirements.txt','script.py']:
    try:
        os.remove(f'./{str_file}')
    except:
        pass